# Video Processing for License Plate Detection

Install the required dependencies

In [1]:
!pip install ultralytics==8.3.152 easyocr==1.7.2 transformers==4.52.4 opencv-python==4.11.0.86 pyyaml>=6.0

  Add necessary imports

In [3]:
import os
import cv2
import sys
import json
import time
import torch
import numpy as np
from pathlib import Path
from typing import Optional, List, Dict, Tuple
from PIL import Image
from datetime import datetime
from ultralytics import YOLO
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

# Google colab imports
from google.colab import drive

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


#### Mount the drive path which contains the models and the video in the following format-

FOLDER \
|___ Models \
&emsp;&emsp;|____ best_license_plate_model.pt \
|___ Videos \
&emsp;&emsp;&emsp;|___ Video_file.mp4

In [4]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


### Initialize the global variables

In [5]:
# Path to the Video Source
VIDEO_SOURCE: str = (
    r"/content/drive/MyDrive/DATA/Videos/fog.mp4"
)
# Output path to save the processed video
OUTPUT_VIDEO_PATH: str = (
    f"/content/drive/MyDrive/DATA/Videos/processed_video_{datetime.now().strftime('%Y%m%d%H%M%S')}.mp4"
)
# YOLO Model Path
MODEL_PATH = Path(
    r"/content/drive/MyDrive/DATA/Models/best_license_plate_model.pt"
)
# For faster computation, use FP16
FP16_ENABLED = True
# Resize dimensions
RESIZE_DIMENSIONS: Tuple[int, int] = (1280, 720)
# How long to keep plates visible (in frames)
PLATE_PERSISTENCE_FRAMES: int = 150
# Minimum frames between same plate detections to consider it "new"
UNIQUE_DETECTION_COOLDOWN: int = 30
# Show/hide the persistent detection panel (True = show, False = hide)
SHOW_DETECTION_PANEL: bool = True

### Video Analyzer class

In [7]:
class LicensePlateVideoAnalyzer:
    def __init__(self):

        # Setup GPU if available
        self._setup_gpu()

        # Load models
        self._load_models()

        # Initialize tracking
        self.detections = []
        self.tracked_plates = (
            {}
        )  # Track unique plates: {plate_text: {last_frame, first_frame, count, bbox}}
        self.recent_detections = []  # Recently detected plates for display

        print("LicensePlateVideoAnalyzer initialized successfully.")

    def _setup_gpu(self):
        self.device = "cpu"
        self.use_gpu = False

        if torch.cuda.is_available():
            self.device = "cuda"
            self.use_gpu = True

            gpu_name = torch.cuda.get_device_name(0)
            gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
            print(f"Using GPU: {gpu_name} with {gpu_memory:.2f} GB memory")
        else:
            print("Using CPU for processing")

    def _load_models(self):
        """Load YOLO and TrOCR models."""

        model_path = Path(MODEL_PATH)

        if not model_path.exists():
            print(f"Model file not found at {model_path}")
            sys.exit(1)

        print(f"Loading YOLO model from {model_path}")
        self.yolo_model = YOLO(str(model_path))

        if self.use_gpu:
            self.yolo_model.to(self.device)

        print(f" YOLO model loaded (classes: {self.yolo_model.names})")

        print("Loading TrOCR model and processor")
        trocr_model_name = "microsoft/trocr-base-printed"
        print(f"Loading TrOCR model: {trocr_model_name}")
        self.processor = TrOCRProcessor.from_pretrained(trocr_model_name, use_fast=True)
        print(f"Loading TrOCR vision encoder model")
        self.trocr_model = VisionEncoderDecoderModel.from_pretrained(trocr_model_name)

        # Move TrOCR model to GPU
        if self.use_gpu:
            self.trocr_model.to(self.device)

            # Use FP16 if enabled
            if FP16_ENABLED:
                self.trocr_model.half()

        print("TrOCR model and processor loaded successfully")

    def _run_ocr(self, cropped_plate: np.ndarray) -> str:
        """Run OCR on cropped license plate"""
        try:
            if len(cropped_plate.shape) == 2:
                pil_img = Image.fromarray(cropped_plate)
            else:
                pil_img = Image.fromarray(
                    cv2.cvtColor(cropped_plate, cv2.COLOR_BGR2RGB)
                )

            # Process with TrOCR
            inputs = self.processor(images=pil_img, return_tensors="pt").pixel_values

            if self.use_gpu:
                inputs = inputs.to(self.device)
                if FP16_ENABLED:
                    inputs = inputs.half()

            with torch.no_grad():
                outputs = self.trocr_model.generate(inputs)

            text = self.processor.batch_decode(outputs, skip_special_tokens=True)[0]
            return text.strip()

        except Exception as e:
            return "OCR_ERROR"

    def _draw_detection_panel(self, frame: np.ndarray, frame_num: int) -> np.ndarray:
        """Draw a persistent panel showing recent license plate detections"""
        # Update recent detections - remove old ones
        self.recent_detections = [
            det
            for det in self.recent_detections
            if frame_num - det["last_seen"] < PLATE_PERSISTENCE_FRAMES
        ]

        if not self.recent_detections:
            return frame

        # Panel configuration
        panel_width = 400
        panel_x = frame.shape[1] - panel_width - 10
        panel_y = 10
        line_height = 45

        # Draw semi-transparent background panel
        overlay = frame.copy()
        panel_height = min(
            len(self.recent_detections) * line_height + 40, frame.shape[0] - 20
        )
        cv2.rectangle(
            overlay,
            (panel_x, panel_y),
            (panel_x + panel_width, panel_y + panel_height),
            (0, 0, 0),
            -1,
        )
        cv2.addWeighted(overlay, 0.7, frame, 0.3, 0, frame)

        # Draw title
        cv2.putText(
            frame,
            "DETECTED PLATES",
            (panel_x + 10, panel_y + 25),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2,
        )

        # Draw each recent detection
        y_offset = panel_y + 50
        for i, det in enumerate(self.recent_detections[:10]):  # Show max 10
            plate_text = det["ocr_text"]
            conf = det["confidence"]
            age = frame_num - det["last_seen"]

            # Color based on freshness (green = fresh, yellow = old)
            freshness = 1 - (age / PLATE_PERSISTENCE_FRAMES)
            color = (0, int(255 * freshness), int(255 * (1 - freshness)))

            # Draw plate text with large font
            text = f"{plate_text}"
            cv2.putText(
                frame,
                text,
                (panel_x + 15, y_offset),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,
                color,
                2,
            )

            # Draw confidence and count
            subtext = f"Conf: {conf:.2f} | Seen: {det['count']}x"
            cv2.putText(
                frame,
                subtext,
                (panel_x + 15, y_offset + 20),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.4,
                (200, 200, 200),
                1,
            )

            y_offset += line_height

        return frame

    def _process_frame(
        self, frame: np.ndarray, frame_num: int
    ) -> Tuple[np.ndarray, List[Dict]]:
        """Process single frame for license plate detection and OCR"""
        ann_frame = frame.copy()
        frame_detections = []

        # Run YOLO inference
        inf_start = time.time()
        results = self.yolo_model(frame, verbose=False)
        inf_time = time.time() - inf_start

        ocr_start = time.time()
        detection_count = 0

        min_crop_size = [40, 15]
        max_crop_size = [400, 150]

        for result in results:
            boxes = result.boxes

            for box in boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                confidence = float(box.conf[0].item())
                class_id = int(box.cls[0].item())
                detected_class = self.yolo_model.names.get(class_id, "Unknown")

                # Filter by class and confidence
                if detected_class == "LP" and confidence >= 0.7:
                    # Crop license plate region
                    cropped_plate = frame[y1:y2, x1:x2]

                    # Check crop size
                    if (
                        cropped_plate.shape[0] < min_crop_size[1]
                        or cropped_plate.shape[1] < min_crop_size[0]
                    ):
                        continue

                    # Run OCR
                    ocr_text = self._run_ocr(cropped_plate)

                    # Update tracking for this plate
                    is_new = False
                    if ocr_text in self.tracked_plates:
                        # Update existing plate
                        if (
                            frame_num - self.tracked_plates[ocr_text]["last_frame"]
                            > UNIQUE_DETECTION_COOLDOWN
                        ):
                            is_new = True
                        self.tracked_plates[ocr_text]["last_frame"] = frame_num
                        self.tracked_plates[ocr_text]["count"] += 1
                        self.tracked_plates[ocr_text]["bbox"] = [x1, y1, x2, y2]
                    else:
                        # New plate detected
                        is_new = True
                        self.tracked_plates[ocr_text] = {
                            "first_frame": frame_num,
                            "last_frame": frame_num,
                            "count": 1,
                            "bbox": [x1, y1, x2, y2],
                            "confidence": confidence,
                        }

                    # Add/update in recent detections
                    existing = next(
                        (
                            d
                            for d in self.recent_detections
                            if d["ocr_text"] == ocr_text
                        ),
                        None,
                    )
                    if existing:
                        existing["last_seen"] = frame_num
                        existing["confidence"] = max(existing["confidence"], confidence)
                        existing["count"] += 1
                    else:
                        self.recent_detections.insert(
                            0,
                            {
                                "ocr_text": ocr_text,
                                "confidence": confidence,
                                "last_seen": frame_num,
                                "count": 1,
                            },
                        )

                    # Draw bounding box with thicker line for current detection
                    box_color = (
                        (0, 255, 255) if is_new else (0, 255, 0)
                    )  # Yellow for new, green for existing
                    box_thickness = 3 if is_new else 2
                    cv2.rectangle(
                        ann_frame, (x1, y1), (x2, y2), box_color, box_thickness
                    )

                    # Prepare label with larger text
                    label = f"{ocr_text}"
                    conf_label = f"{confidence:.2f}"

                    # Draw label background
                    font = cv2.FONT_HERSHEY_SIMPLEX
                    font_scale = 1.2 if is_new else 0.8
                    thickness = 3 if is_new else 2

                    (text_w, text_h), _ = cv2.getTextSize(
                        label, font, font_scale, thickness
                    )
                    (conf_w, conf_h), _ = cv2.getTextSize(conf_label, font, 0.5, 1)

                    # Main label background
                    bg_y1 = max(y1 - text_h - 15, 0)
                    cv2.rectangle(
                        ann_frame,
                        (x1, bg_y1),
                        (x1 + text_w + 10, y1),
                        box_color,
                        -1,
                    )

                    # Draw the plate text
                    cv2.putText(
                        ann_frame,
                        label,
                        (x1 + 5, y1 - 5),
                        font,
                        font_scale,
                        (0, 0, 0),
                        thickness,
                    )

                    # Draw confidence below
                    cv2.putText(
                        ann_frame,
                        conf_label,
                        (x1, y2 + 15),
                        font,
                        0.5,
                        box_color,
                        1,
                    )

                    # Add "NEW!" indicator for new detections
                    if is_new:
                        cv2.putText(
                            ann_frame,
                            "NEW!",
                            (x2 - 60, y1 - 10),
                            font,
                            0.7,
                            (0, 255, 255),
                            2,
                        )

                    # Store detection
                    detection = {
                        "frame_number": frame_num,
                        "timestamp": datetime.now().isoformat(),
                        "bbox": [x1, y1, x2, y2],
                        "confidence": confidence,
                        "ocr_text": ocr_text,
                        "class": detected_class,
                        "is_new": is_new,
                    }
                    frame_detections.append(detection)
                    detection_count += 1

        ocr_time = time.time() - ocr_start

        # Draw the persistent detection panel (if enabled)
        if SHOW_DETECTION_PANEL:
            ann_frame = self._draw_detection_panel(ann_frame, frame_num)

        return ann_frame, frame_detections

    def process_video(
        self, video_source: str, output_path: Optional[str] = None
    ) -> str:  # type: ignore
        """
        Process video with license plate detection
        """
        if not video_source:
            raise ValueError("Video source path is empty.")

        print(f"Processing video: {video_source}")

        reader = cv2.VideoCapture(video_source)

        if not reader.isOpened():
            raise ValueError(f"Failed to open video source: {video_source}")

        self.fps = int(reader.get(cv2.CAP_PROP_FPS))
        self.frame_count = int(reader.get(cv2.CAP_PROP_FRAME_COUNT))
        self.width = int(reader.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.height = int(reader.get(cv2.CAP_PROP_FRAME_HEIGHT))

        if RESIZE_DIMENSIONS:
            self.width, self.height = RESIZE_DIMENSIONS

        self.current_frame = 0

        output_path = Path(OUTPUT_VIDEO_PATH)  # type: ignore

        if not os.path.exists(output_path.parent):
            os.makedirs(output_path.parent)

        fourcc = cv2.VideoWriter.fourcc(*"mp4v")
        writer = cv2.VideoWriter(
            str(output_path), fourcc, self.fps, (self.width, self.height)
        )

        if not writer.isOpened():
            raise ValueError(f"Failed to open video writer for output: {output_path}")

        print(f"Output video will be saved to: {output_path}")

        frame_skip = 1
        frame_num = 0
        processed_frames = 0
        self.detections = []

        try:
            while True:
                ret, frame = reader.read()
                if not ret:
                    break

                frame_num += 1

                # Skip frames if needed
                if (frame_num - 1) % frame_skip != 0:
                    continue

                # Resize frame if needed
                if RESIZE_DIMENSIONS:
                    frame = cv2.resize(frame, RESIZE_DIMENSIONS)

                annotated_frame, frame_detections = self._process_frame(
                    frame, frame_num
                )
                self.detections.extend(frame_detections)

                # Resize annotated frame to match writer dimensions
                if (
                    annotated_frame.shape[1] != self.width
                    or annotated_frame.shape[0] != self.height
                ):
                    annotated_frame = cv2.resize(
                        annotated_frame, (self.width, self.height)
                    )

                # Write annotated frame to output video
                writer.write(annotated_frame)

                processed_frames += 1

                if processed_frames % 50 == 0:
                    print(f"Processed {processed_frames} frames...")

        finally:
            reader.release()
            writer.release()

        print(
            f"Video processing completed. Processed {processed_frames} frames in total."
        )

        return str(output_path)

## Main function

In [8]:
def main():
    # Diagnostic check for video source existence
    if not os.path.exists(VIDEO_SOURCE):
        print(f"Error: Video file not found at {VIDEO_SOURCE}.")
        print("Please verify the path and ensure the file exists in your Google Drive.")
        video_dir = os.path.dirname(VIDEO_SOURCE)
        if os.path.exists(video_dir):
            print(f"Contents of video directory '{video_dir}': {os.listdir(video_dir)}")
        else:
            print(f"Video directory '{video_dir}' does not exist.")
        return

    analyzer = LicensePlateVideoAnalyzer()
    output_video = analyzer.process_video(VIDEO_SOURCE, OUTPUT_VIDEO_PATH)
    print(f"Processed video saved at: {output_video}")


if __name__ == "__main__":
    main()


Using GPU: Tesla T4 with 14.56 GB memory
Loading YOLO model from /content/drive/MyDrive/DATA/Models/best_license_plate_model.pt
 YOLO model loaded (classes: {0: 'LP'})
Loading TrOCR model and processor
Loading TrOCR model: microsoft/trocr-base-printed


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

TrOCR model and processor loaded successfully
LicensePlateVideoAnalyzer initialized successfully.
Processing video: /content/drive/MyDrive/DATA/Videos/fog.mp4
Output video will be saved to: /content/drive/MyDrive/DATA/Videos/processed_video_20260504174648.mp4
Processed 50 frames...
Processed 100 frames...
Processed 150 frames...
Processed 200 frames...
Processed 250 frames...
Processed 300 frames...
Processed 350 frames...
Processed 400 frames...
Processed 450 frames...
Processed 500 frames...
Processed 550 frames...
Processed 600 frames...
Processed 650 frames...
Processed 700 frames...
Processed 750 frames...
Processed 800 frames...
Processed 850 frames...
Processed 900 frames...
Processed 950 frames...
Processed 1000 frames...
Processed 1050 frames...
Processed 1100 frames...
Processed 1150 frames...
Processed 1200 frames...
Processed 1250 frames...
Processed 1300 frames...
Processed 1350 frames...
Processed 1400 frames...
Processed 1450 frames...
Video processing completed. Process